# Gridlock Hackathon 2.0 - End-to-End Solution

**Team:** APIcalypse Now

This notebook provides a complete walkthrough for traffic demand prediction: data loading, quality checks, EDA, feature engineering, chronological validation, model training, and submission generation.

## 1. Setup

Place `train.csv`, `test.csv`, and `sample_submission.csv` in the `data/` directory before running this notebook.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

sys.path.append(str(SRC_DIR))
from gridlock_solution import (
    add_time_features,
    TunedBlendForecaster,
    DailyLagForecaster,
    r2_score_np,
    rmse_np,
    mae_np,
)

## 2. Data Loading

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

print("train", train.shape)
print("test", test.shape)
print("sample", sample_submission.shape)
train.head()

## 3. Data Quality Report

We inspect missing values, duplicates, target range, and train/test coverage.

In [ ]:
quality = pd.DataFrame({
    "train_missing": train.isna().sum(),
    "test_missing": test.isna().sum() if set(test.columns).issubset(set(train.columns)) else pd.Series(dtype=float),
})
print(quality)

print("Train duplicates:", train.duplicated().sum())
print("Test duplicates:", test.duplicated().sum())
print(train["demand"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))
print("Target outside [0, 1]:", ((train["demand"] < 0) | (train["demand"] > 1)).sum())

## 4. Split Analysis

The most important insight is the chronological split: day 48 is complete, day 49 has early labels, and test contains future day-49 slots.

In [ ]:
train_t = add_time_features(train)
test_t = add_time_features(test)

print("Train day ranges")
print(train_t.groupby("day")["minute"].agg(["min", "max", "nunique", "count"]))

print("Test day ranges")
print(test_t.groupby("day")["minute"].agg(["min", "max", "nunique", "count"]))

## 5. EDA Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(train["demand"], bins=50, color="#2563eb", alpha=0.85)
axes[0].set_title("Demand Distribution")
axes[0].set_xlabel("Demand")
axes[0].set_ylabel("Rows")

road_means = train.groupby("RoadType", dropna=False)["demand"].mean().sort_values(ascending=False)
road_means.plot(kind="bar", ax=axes[1], color="#16a34a")
axes[1].set_title("Mean Demand by Road Type")
axes[1].set_ylabel("Mean demand")
plt.tight_layout()
plt.show()

In [ ]:
time_profile = train_t.groupby("minute")["demand"].mean()
plt.figure(figsize=(12, 4))
plt.plot(time_profile.index, time_profile.values, color="#f97316")
plt.title("Average Demand by Time of Day")
plt.xlabel("Minute of day")
plt.ylabel("Mean demand")
plt.grid(alpha=0.25)
plt.show()

## 6. Feature Engineering

The final solution relies on temporal, lag, road-context, and geohash-hour priors. The core engineered signals are implemented in `src/gridlock_solution.py`.

In [ ]:
model = TunedBlendForecaster(
    geo_hour_weight=0.45,
    lag_weight=0.05,
    road_weight=0.50,
    minute_weight=0.0,
    calibration_decay_minutes=720.0,
)

## 7. Chronological Validation

We hold out later known day49 records and calibrate only on earlier day49 records. This mimics the test task more closely than random validation.

In [ ]:
cutoff = 90
train_t = add_time_features(train)
valid_mask = (train_t["day"] == 49) & (train_t["minute"] > cutoff)
train_part = train.loc[~valid_mask].copy()
valid_part = train.loc[valid_mask].copy()

model.fit(train_part, calibration_cutoff=cutoff)
valid_pred = model.predict(valid_part)

print("R2:", r2_score_np(valid_part["demand"], valid_pred))
print("RMSE:", rmse_np(valid_part["demand"], valid_pred))
print("MAE:", mae_np(valid_part["demand"], valid_pred))

## 8. Model Comparison

In [ ]:
baseline = DailyLagForecaster().fit(train_part, calibration_cutoff=cutoff)
baseline_pred = baseline.predict(valid_part)

comparison = pd.DataFrame([
    {"model": "calibrated_daily_lag", "r2": r2_score_np(valid_part["demand"], baseline_pred), "rmse": rmse_np(valid_part["demand"], baseline_pred), "mae": mae_np(valid_part["demand"], baseline_pred)},
    {"model": "rolling_cv_road_geo_blend", "r2": r2_score_np(valid_part["demand"], valid_pred), "rmse": rmse_np(valid_part["demand"], valid_pred), "mae": mae_np(valid_part["demand"], valid_pred)},
])
comparison.sort_values("r2", ascending=False)

## 9. Train Final Model and Generate Submission

In [ ]:
final_model = TunedBlendForecaster(
    geo_hour_weight=0.45,
    lag_weight=0.05,
    road_weight=0.50,
    minute_weight=0.0,
    calibration_decay_minutes=720.0,
).fit(train)

final_pred = final_model.predict(test)
submission = pd.DataFrame({"Index": test["Index"].values, "demand": np.clip(final_pred, 0, 1)})
submission.to_csv(OUTPUT_DIR / "submission.csv", index=False)
submission.head()

## 10. Key Takeaways

- Chronological validation is essential.
- Road type and geohash-hour priors are stronger than generic random splits suggest.
- Early day49 labels provide valuable same-day calibration.
- Leaderboard feedback favored a road-context-heavy blend with persistent calibration.